In [1]:
from contextlib import contextmanager
import sys
from pathlib import Path
from typing import Iterable, Iterator

import pandas as pd
import shutil
import tempfile
from urllib.request import urlopen, Request
import zipfile
import os

for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    source_dir = path / "src"
    if source_dir.exists():
        sys.path.append(str(source_dir))
        break
else:
    raise FileNotFoundError("Nao encontrei a pasta src nos diretorios pais.")

from nofis_classifier.paths import find_project_root

In [3]:
BASE_URL = os.getenv("BASE_URL")
DATASET_NAME = os.getenv("DATASET_NAME")


DEFAULT_CSV_KWARGS = {
    "sep": ";",
    "encoding": "latin1",
    "low_memory": False,
}
# Forçar colunas a serem String
DEFAULT_TEXT_COLUMNS = [
    "CHAVE DE ACESSO",
]

In [11]:
# Esta etapa nao seleciona colunas nem reduz duplicatas.
# Ela baixa o ZIP mensal, concatena os CSVs extraidos e salva tudo em Parquet.

# Funções

## Baixar arquivo Zip da fonte de Notas Fiscais

In [5]:
def formatar_competencia(ano: int, mes: int) -> tuple[str, str]:
    """Return competencia values as yyyyMM and yyyy-MM."""
    ano = int(ano)
    mes = int(mes)

    if not 1 <= mes <= 12:
        raise ValueError("mes deve estar entre 1 e 12.")

    return f"{ano:04d}{mes:02d}", f"{ano:04d}-{mes:02d}"


def dataset_dir(stage: str, dataset_name: str = DATASET_NAME) -> Path:
    """Return data/<stage>/<dataset_name>, creating it when needed."""
    path = find_project_root() / "data" / stage / dataset_name
    path.mkdir(parents=True, exist_ok=True)
    return path


def download_file(url: str, destination: Path, chunk_size: int = 1024 * 1024) -> None:
    """Download a URL to destination, using requests when available."""
    try:
        import requests
    except ImportError:
        request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urlopen(request) as response, destination.open("wb") as file:
            while chunk := response.read(chunk_size):
                file.write(chunk)
        return

    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with destination.open("wb") as file:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    file.write(chunk)


def safe_extract_zip(zip_path: Path, destination: Path) -> None:
    """Extract zip_path without allowing files to escape destination."""
    destination = destination.resolve()

    with zipfile.ZipFile(zip_path) as zip_file:
        for member in zip_file.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f"Arquivo inseguro dentro do zip: {member.filename}")

        zip_file.extractall(destination)


def baixar_zip_notas_fiscais(
    ano: int,
    mes: int,
    raw_dir: Path | str | None = None,
    overwrite: bool = True,
) -> Path:
    """Baixa o zip bruto em data/raw/notas-fiscais/yyyy-MM.zip."""
    competencia, competencia_com_hifen = formatar_competencia(ano, mes)
    url = f"{BASE_URL}/{competencia}"
    raw_path = Path(raw_dir) if raw_dir is not None else dataset_dir("raw")
    raw_path.mkdir(parents=True, exist_ok=True)
    destino_final = raw_path / f"{competencia_com_hifen}.zip"

    if destino_final.exists() and not overwrite:
        print(f"Zip ja existe: {destino_final}")
        return destino_final

    with tempfile.TemporaryDirectory() as tmp:
        zip_path = Path(tmp) / f"{competencia}.zip"
        print(f"Baixando {url}...")
        download_file(url, zip_path)

        if destino_final.exists():
            destino_final.unlink()
        shutil.move(str(zip_path), str(destino_final))

    print(f"Zip salvo em: {destino_final}")
    return destino_final


def listar_arquivos_zip(zip_path: Path | str) -> list[str]:
    """Lista os arquivos contidos em um zip."""
    with zipfile.ZipFile(zip_path) as zip_file:
        return zip_file.namelist()

## Extrair CSVs e converter para Parquet

In [6]:
@contextmanager
def extrair_zip_temporario(zip_path: Path | str) -> Iterator[Path]:
    """Extrai um zip em pasta temporaria e limpa tudo ao sair do bloco."""
    with tempfile.TemporaryDirectory() as tmp:
        extract_path = Path(tmp) / "extracted"
        extract_path.mkdir()
        safe_extract_zip(Path(zip_path), extract_path)
        yield extract_path


def encontrar_csvs(pasta: Path | str) -> list[Path]:
    """Encontra todos os CSVs extraidos."""
    csvs = sorted(Path(pasta).rglob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"Nao encontrei arquivos CSV em {pasta}.")
    return csvs


def ler_csvs_notas_fiscais(
    pasta_extraida: Path | str,
    csv_kwargs: dict | None = None,
    colunas_texto: list[str] | None = None,
) -> pd.DataFrame:
    """Le todos os CSVs extraidos e concatena em um unico DataFrame."""
    kwargs = DEFAULT_CSV_KWARGS.copy()
    if csv_kwargs:
        kwargs.update(csv_kwargs)

    dtype = dict(kwargs.pop("dtype", {}) or {})
    for coluna in (colunas_texto or DEFAULT_TEXT_COLUMNS):
        dtype.setdefault(coluna, "string")
    kwargs["dtype"] = dtype

    dataframes = []
    for csv_path in encontrar_csvs(pasta_extraida):
        df = pd.read_csv(csv_path, **kwargs)
        df["arquivo_origem"] = csv_path.name
        dataframes.append(df)

    return pd.concat(dataframes, ignore_index=True)


def baixar_e_converter_mes_notas_fiscais(
    ano: int,
    mes: int,
    overwrite_raw: bool = True,
    overwrite_interim: bool = True,
    csv_kwargs: dict | None = None,
    colunas_texto: list[str] | None = None,
) -> Path:
    """Baixa o zip mensal, le os CSVs extraidos e salva o conteudo completo em Parquet."""
    _, competencia_com_hifen = formatar_competencia(ano, mes)
    interim_path = dataset_dir("interim") / f"{competencia_com_hifen}.parquet"

    if interim_path.exists() and not overwrite_interim:
        print(f"Parquet ja existe: {interim_path}")
        return interim_path

    zip_path = baixar_zip_notas_fiscais(
        ano,
        mes,
        overwrite=overwrite_raw,
    )

    print(f"Extraindo temporariamente {zip_path.name}...")
    with extrair_zip_temporario(zip_path) as pasta_extraida:
        df_notas = ler_csvs_notas_fiscais(
            pasta_extraida,
            csv_kwargs=csv_kwargs,
            colunas_texto=colunas_texto,
        )

    if interim_path.exists():
        interim_path.unlink()
    df_notas.to_parquet(interim_path, index=False)

    print(f"Parquet salvo em: {interim_path}")
    return interim_path


def baixar_e_converter_periodos_notas_fiscais(
    periodos: Iterable[tuple[int, int]],
    overwrite_raw: bool = True,
    overwrite_interim: bool = True,
    csv_kwargs: dict | None = None,
    colunas_texto: list[str] | None = None,
) -> list[Path]:
    """Baixa e converte uma lista de periodos no formato (ano, mes)."""
    parquet_paths = []

    for ano, mes in periodos:
        parquet_path = baixar_e_converter_mes_notas_fiscais(
            ano,
            mes,
            overwrite_raw=overwrite_raw,
            overwrite_interim=overwrite_interim,
            csv_kwargs=csv_kwargs,
            colunas_texto=colunas_texto,
        )
        parquet_paths.append(parquet_path)

    return parquet_paths


# Main

Colunas disponíveis:
```
Index(['CHAVE DE ACESSO', 'MODELO', 'SÉRIE', 'NÚMERO', 'NATUREZA DA OPERAÇÃO',
       'DATA EMISSÃO', 'CPF/CNPJ Emitente', 'RAZÃO SOCIAL EMITENTE',
       'INSCRIÇÃO ESTADUAL EMITENTE', 'UF EMITENTE', 'MUNICÍPIO EMITENTE',
       'CÓDIGO ÓRGÃO SUPERIOR DESTINATÁRIO', 'ÓRGÃO SUPERIOR DESTINATÁRIO',
       'CÓDIGO ÓRGÃO DESTINATÁRIO', 'ÓRGÃO DESTINATÁRIO', 'CNPJ DESTINATÁRIO',
       'NOME DESTINATÁRIO', 'UF DESTINATÁRIO', 'INDICADOR IE DESTINATÁRIO',
       'DESTINO DA OPERAÇÃO', 'CONSUMIDOR FINAL', 'PRESENÇA DO COMPRADOR',
       'NÚMERO PRODUTO', 'DESCRIÇÃO DO PRODUTO/SERVIÇO', 'CÓDIGO NCM/SH',
       'NCM/SH (TIPO DE PRODUTO)', 'CFOP', 'QUANTIDADE', 'UNIDADE',
       'VALOR UNITÁRIO', 'VALOR TOTAL'],
      dtype='str')
```

In [7]:
ano = 2025
mes = 4

In [8]:
# primeiro baixa apenas o ZIP bruto.
zip_path = baixar_zip_notas_fiscais(ano, mes)
listar_arquivos_zip(zip_path)[:10]

Baixando https://portaldatransparencia.gov.br/download-de-dados/notas-fiscais/202504...
Zip salvo em: C:\Users\samue\Scripts\Nofis-Classifier\data\raw\items-notas-fiscais\2025-04.zip


['202504_NFe_NotaFiscal.csv',
 '202504_NFe_NotaFiscalEvento.csv',
 '202504_NFe_NotaFiscalItem.csv']

In [9]:
baixar_e_converter_mes_notas_fiscais(ano, mes)

Baixando https://portaldatransparencia.gov.br/download-de-dados/notas-fiscais/202504...
Zip salvo em: C:\Users\samue\Scripts\Nofis-Classifier\data\raw\items-notas-fiscais\2025-04.zip
Extraindo temporariamente 2025-04.zip...
Parquet salvo em: C:\Users\samue\Scripts\Nofis-Classifier\data\interim\items-notas-fiscais\2025-04.parquet


WindowsPath('C:/Users/samue/Scripts/Nofis-Classifier/data/interim/items-notas-fiscais/2025-04.parquet')

In [10]:
periodos = [
    (2025, 1),
    (2025, 2),
    (2025, 3),
    (2025, 4),
]

baixar_e_converter_periodos_notas_fiscais(periodos)

Baixando https://portaldatransparencia.gov.br/download-de-dados/notas-fiscais/202501...
Zip salvo em: C:\Users\samue\Scripts\Nofis-Classifier\data\raw\items-notas-fiscais\2025-01.zip
Extraindo temporariamente 2025-01.zip...
Parquet salvo em: C:\Users\samue\Scripts\Nofis-Classifier\data\interim\items-notas-fiscais\2025-01.parquet
Baixando https://portaldatransparencia.gov.br/download-de-dados/notas-fiscais/202502...
Zip salvo em: C:\Users\samue\Scripts\Nofis-Classifier\data\raw\items-notas-fiscais\2025-02.zip
Extraindo temporariamente 2025-02.zip...
Parquet salvo em: C:\Users\samue\Scripts\Nofis-Classifier\data\interim\items-notas-fiscais\2025-02.parquet
Baixando https://portaldatransparencia.gov.br/download-de-dados/notas-fiscais/202503...
Zip salvo em: C:\Users\samue\Scripts\Nofis-Classifier\data\raw\items-notas-fiscais\2025-03.zip
Extraindo temporariamente 2025-03.zip...
Parquet salvo em: C:\Users\samue\Scripts\Nofis-Classifier\data\interim\items-notas-fiscais\2025-03.parquet
Baixand

[WindowsPath('C:/Users/samue/Scripts/Nofis-Classifier/data/interim/items-notas-fiscais/2025-01.parquet'),
 WindowsPath('C:/Users/samue/Scripts/Nofis-Classifier/data/interim/items-notas-fiscais/2025-02.parquet'),
 WindowsPath('C:/Users/samue/Scripts/Nofis-Classifier/data/interim/items-notas-fiscais/2025-03.parquet'),
 WindowsPath('C:/Users/samue/Scripts/Nofis-Classifier/data/interim/items-notas-fiscais/2025-04.parquet')]